# Quick Start: PSO Segmentation

Welcome! This notebook demonstrates the simplest way to use **pso-segmentation** for segmenting a continuous variable with constraints.

We'll:
1. Generate sample data (credit scoring is the example)
2. Run segmentation with a simple fitness function
3. Examine results and interpret metrics

## Setup

Import required libraries and generate sample data:

In [1]:
import sys
from pathlib import Path

# Add src to path for importing pso_segmentation
sys.path.insert(0, str(Path("..") / "src"))

import numpy as np
import pandas as pd

from pso_segmentation import (
    OptimizerConfig,
    SegmentationOptimizer,
    example_fitness_r2_only,
)

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully!")

Libraries imported successfully!


## Generate Sample Data

Create synthetic data with a score and a binary target (default is the example):

In [2]:
# Create 1000 samples with a continuous score
n_samples = 1000
scores = np.random.beta(a=2, b=5, size=n_samples)  # Skewed distribution for demonstration

# Generate binary target: more likely for higher scores
labels = (np.random.rand(n_samples) < scores).astype(int)

# Create DataFrame for inspection (default is the example target)
df = pd.DataFrame({"score": scores, "default": labels})

print(f"Dataset: {len(df)} customers")
print("\nScore distribution:")
print(df["score"].describe())
print(f"\nTarget mean (PD): {df['default'].mean():.1%}")
print("\nFirst few rows:")
df.head()

Dataset: 1000 customers

Score distribution:
count    1000.000000
mean        0.285676
std         0.155269
min         0.007248
25%         0.165606
50%         0.262671
75%         0.389635
max         0.813342
Name: score, dtype: float64

Target mean (PD): 28.4%

First few rows:


,score,default
0,0.353677,0
1,0.248558,0
2,0.415959,1
3,0.159968,0
4,0.550283,1


## Run Segmentation

Segment the customers into risk groups using PSO optimization:

In [ ]:
# Run Segmentation (using OO API to access cuts)


def fitness(cuts):
    return example_fitness_r2_only(cuts, scores, labels)


config = OptimizerConfig(n_segments=4, pop_size=80, max_iter=300, seed=42)
optimizer = SegmentationOptimizer(config)
optimizer.fit(scores, labels, fitness)
result = optimizer.get_metrics()
cuts = optimizer.get_cuts()

print("Segmentation completed!")
print(f"\nBest R²: {result.r2:.4f}")
print(f"Number of segments: {result.n_segments}")

Segmentation completed!

Best R²: 0.1308
Number of segments: 4
Cuts: [0.19138226 0.44516378 0.52955615]


## Inspect Results

Examine the segmentation output:

In [6]:
print("=" * 60)
print("SEGMENTATION RESULTS")
print("=" * 60)

print("\nSegment Cut Values:")
for i, cut in enumerate(cuts):
    print(f"  Cut {i}: {cut:.4f}")

print("\nTarget mean (PD) by Segment:")
for i, segment_mean in enumerate(result.target_mean_by_segment):
    print(f"  Segment {i + 1}: {segment_mean * 100:.1f}%")

print("\nSegment Sizes:")
for i, size in enumerate(result.segment_sizes):
    pct = size / len(scores) * 100
    print(f"  Segment {i + 1}: {int(size):4d} customers ({pct:.1f}%)")

print("\nQuality Metrics:")
print(f"  R²:          {result.r2:.4f}  (variance explained)")
print(f"  H_inter:     {result.h_inter:.4f}  (between-group separation)")
print(f"  H_intra:     {result.h_intra:.4f}  (within-group homogeneity)")
print(f"  Monotonic:   {result.is_monotonic_increasing()}")
print(f"  Balanced:    {result.is_balanced()}")

SEGMENTATION RESULTS

Segment Cut Values:
  Cut 0: 0.1914
  Cut 1: 0.4452
  Cut 2: 0.5296

Target mean (PD) by Segment:
  Segment 1: 11.3%
  Segment 2: 28.6%
  Segment 3: 51.0%
  Segment 4: 67.5%

Segment Sizes:
  Segment 1:  319 customers (31.9%)
  Segment 2:  503 customers (50.3%)
  Segment 3:   98 customers (9.8%)
  Segment 4:   80 customers (8.0%)

Quality Metrics:
  R²:          0.1308  (variance explained)
  H_inter:     26.5916  (between-group separation)
  H_intra:     176.7524  (within-group homogeneity)
  Monotonic:   True
  Balanced:    False


## Assign Segments to Customers

Add segment assignments to the dataframe:

In [7]:
# Get segment assignments
segments = optimizer.get_segments()

# Create segmented DataFrame
df["segment"] = segments

print("\nSegmented Data (first 10 rows):")
print(df.head(10))

print("\nSegment Assignment Summary:")
print(
    df.groupby("segment")[["score", "default"]]
    .agg({"score": ["min", "max", "mean"], "default": ["sum", "count", "mean"]})
    .round(3)
)


Segmented Data (first 10 rows):
      score  default  segment
0  0.353677        0      1.0
1  0.248558        0      1.0
2  0.415959        1      1.0
3  0.159968        0      0.0
4  0.550283        1      3.0
5  0.110945        0      0.0
6  0.509897        0      2.0
7  0.177270        0      0.0
8  0.198290        0      1.0
9  0.376237        0      1.0

Segment Assignment Summary:
         score               default             
           min    max   mean     sum count   mean
segment                                          
0.0      0.007  0.191  0.120      36   319  0.113
1.0      0.191  0.444  0.302     144   503  0.286
2.0      0.445  0.529  0.482      50    98  0.510
3.0      0.530  0.813  0.601      54    80  0.675


## Interpretation

**What do these results mean?**

1. **R² Score** (~0.XX): The PSO found segments that explain this proportion of target variation
   - Higher R² = better predictive power of segments
   - Typical ranges depend on your data; credit scoring is just one example

2. **Target Mean by Segment (PD here)**: How the target is distributed
   - Segment 0 (lowest risk): Lowest target mean
   - Segment 2 (highest risk): Highest target mean
   - Progression shows the hierarchy is captured

3. **Segment Sizes**: Population distribution
   - Roughly balanced segments are easier to manage
   - Very small segments may be unstable

4. **H_inter & H_intra**:
   - H_inter: How separated segments are (higher = better)
   - H_intra: How homogeneous segments are (higher = better)

**Business Application:**
- Use segment assignment for:
  - Pricing or policy rules
  - Monitoring and early warning
  - Decisioning or prioritization

## Next Steps

🎯 **Explore more advanced topics:**

1. **04_business_use_case.ipynb** - PD segmentation with custom objective function and monitoring

📚 **Documentation:**
- Getting Started: https://pso-segmentation.readthedocs.io/getting_started
- Examples: https://pso-segmentation.readthedocs.io/examples
- API Reference: https://pso-segmentation.readthedocs.io/api